In [1]:
# ==============================================================================
# BTS FLIGHT DELAY PREDICTION USING DEEP LEARNING
# IMPROVED PYTORCH DEEP NEURAL NETWORK
# ==============================================================================

import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)


# ==============================================================================
# 1. REPRODUCIBILITY
# ==============================================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)


# ==============================================================================
# 2. LOAD BTS DATASET
# ==============================================================================

dataset_file_path = "final_dataset_12_columns.csv"

if not os.path.exists(dataset_file_path):

    raise FileNotFoundError(
        f"\nDataset not found: {dataset_file_path}\n"
        "Please place final_dataset_12_columns.csv "
        "in the same folder as this Python file."
    )

print("\nLoading dataset:")
print(dataset_file_path)

df = pd.read_csv(dataset_file_path)

print("Original dataset shape:", df.shape)


# ==============================================================================
# 3. REQUIRED COLUMNS
# ==============================================================================

required_columns = [

    "FL_DATE",
    "AIRLINE",
    "ORIGIN",
    "DEST",
    "CRS_DEP_TIME",
    "DISTANCE",
    "DEP_DELAY",

    "DELAY_DUE_CARRIER",
    "DELAY_DUE_WEATHER",
    "DELAY_DUE_NAS",
    "DELAY_DUE_SECURITY",
    "DELAY_DUE_LATE_AIRCRAFT"
]


missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:

    raise ValueError(
        "\nMissing required columns:\n"
        + "\n".join(missing_columns)
    )


# ==============================================================================
# 4. KEEP ONLY REQUIRED COLUMNS
# ==============================================================================

df = df[required_columns].copy()


# ==============================================================================
# 5. DATA CLEANING
# ==============================================================================

print("\nCleaning dataset...")

df["FL_DATE"] = pd.to_datetime(
    df["FL_DATE"],
    errors="coerce"
)

numeric_columns_to_clean = [

    "CRS_DEP_TIME",
    "DISTANCE",
    "DEP_DELAY",

    "DELAY_DUE_CARRIER",
    "DELAY_DUE_WEATHER",
    "DELAY_DUE_NAS",
    "DELAY_DUE_SECURITY",
    "DELAY_DUE_LATE_AIRCRAFT"
]

for column in numeric_columns_to_clean:

    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )


# Fill delay-cause missing values with zero
# because missing cause generally means no recorded cause.

cause_columns = [

    "DELAY_DUE_CARRIER",
    "DELAY_DUE_WEATHER",
    "DELAY_DUE_NAS",
    "DELAY_DUE_SECURITY",
    "DELAY_DUE_LATE_AIRCRAFT"
]

for column in cause_columns:

    df[column] = df[column].fillna(0)


# Remove rows where important prediction information is missing

df = df.dropna(
    subset=[
        "FL_DATE",
        "AIRLINE",
        "ORIGIN",
        "DEST",
        "CRS_DEP_TIME",
        "DISTANCE",
        "DEP_DELAY"
    ]
)

df = df.reset_index(drop=True)

print(
    "Dataset shape after cleaning:",
    df.shape
)


# ==============================================================================
# 6. CREATE DELAY TARGET
# ==============================================================================

# Delay = departure delay >= 15 minutes

df["delay_target"] = (
    df["DEP_DELAY"] >= 15
).astype(int)


print("\nDelay distribution:")

print(
    df["delay_target"].value_counts()
)

print("\nDelay percentages:")

print(
    df["delay_target"].value_counts(
        normalize=True
    ) * 100
)


# ==============================================================================
# 7. CREATE DELAY CAUSE TARGET
# ==============================================================================

cause_values = df[
    cause_columns
].values

df["cause_target"] = np.argmax(
    cause_values,
    axis=1
)


# ==============================================================================
# 8. FEATURE ENGINEERING
# ==============================================================================

print("\nCreating features...")


# ------------------------------------------------------------------------------
# Date features
# ------------------------------------------------------------------------------

df["fl_month"] = (
    df["FL_DATE"].dt.month
)

df["fl_day"] = (
    df["FL_DATE"].dt.day
)

df["fl_dayofweek"] = (
    df["FL_DATE"].dt.dayofweek
)

df["fl_dayofyear"] = (
    df["FL_DATE"].dt.dayofyear
)

df["fl_weekofyear"] = (
    df["FL_DATE"].dt.isocalendar().week
    .astype(int)
)


# ------------------------------------------------------------------------------
# Departure time
# ------------------------------------------------------------------------------

df["crs_dep_hour"] = (
    df["CRS_DEP_TIME"] // 100
) % 24

df["crs_dep_minute"] = (
    df["CRS_DEP_TIME"] % 100
)


# ------------------------------------------------------------------------------
# Time as continuous minutes
# ------------------------------------------------------------------------------

df["departure_minutes"] = (
    df["crs_dep_hour"] * 60
    +
    df["crs_dep_minute"]
)


# ------------------------------------------------------------------------------
# Cyclical time features
# ------------------------------------------------------------------------------

df["hour_sin"] = np.sin(
    2 * np.pi *
    df["crs_dep_hour"] / 24
)

df["hour_cos"] = np.cos(
    2 * np.pi *
    df["crs_dep_hour"] / 24
)


# ------------------------------------------------------------------------------
# Day-of-week cyclical features
# ------------------------------------------------------------------------------

df["dow_sin"] = np.sin(
    2 * np.pi *
    df["fl_dayofweek"] / 7
)

df["dow_cos"] = np.cos(
    2 * np.pi *
    df["fl_dayofweek"] / 7
)


# ------------------------------------------------------------------------------
# Month cyclical features
# ------------------------------------------------------------------------------

df["month_sin"] = np.sin(
    2 * np.pi *
    df["fl_month"] / 12
)

df["month_cos"] = np.cos(
    2 * np.pi *
    df["fl_month"] / 12
)


# ------------------------------------------------------------------------------
# Weekend
# ------------------------------------------------------------------------------

df["is_weekend"] = (
    df["fl_dayofweek"] >= 5
).astype(int)


# ------------------------------------------------------------------------------
# Peak hour
# ------------------------------------------------------------------------------

df["is_peak_hour"] = (
    (
        (df["crs_dep_hour"] >= 7)
        &
        (df["crs_dep_hour"] <= 9)
    )
    |
    (
        (df["crs_dep_hour"] >= 16)
        &
        (df["crs_dep_hour"] <= 19)
    )
).astype(int)


# ------------------------------------------------------------------------------
# Morning / afternoon / evening / night
# ------------------------------------------------------------------------------

df["is_morning"] = (
    (df["crs_dep_hour"] >= 5)
    &
    (df["crs_dep_hour"] < 12)
).astype(int)

df["is_afternoon"] = (
    (df["crs_dep_hour"] >= 12)
    &
    (df["crs_dep_hour"] < 17)
).astype(int)

df["is_evening"] = (
    (df["crs_dep_hour"] >= 17)
    &
    (df["crs_dep_hour"] < 22)
).astype(int)

df["is_night"] = (
    (df["crs_dep_hour"] >= 22)
    |
    (df["crs_dep_hour"] < 5)
).astype(int)


# ------------------------------------------------------------------------------
# Distance features
# ------------------------------------------------------------------------------

df["distance_log"] = np.log1p(
    df["DISTANCE"].clip(lower=0)
)


# ==============================================================================
# 9. CREATE ROUTE FEATURE
# ==============================================================================

df["ROUTE"] = (
    df["ORIGIN"].astype(str)
    +
    "_"
    +
    df["DEST"].astype(str)
)


# ==============================================================================
# 10. FEATURES USED BY THE MODEL
# ==============================================================================

categorical_columns = [

    "AIRLINE",
    "ORIGIN",
    "DEST",
    "ROUTE"
]


numerical_columns = [

    "DISTANCE",
    "CRS_DEP_TIME",

    "fl_month",
    "fl_day",
    "fl_dayofweek",
    "fl_dayofyear",
    "fl_weekofyear",

    "crs_dep_hour",
    "crs_dep_minute",
    "departure_minutes",

    "hour_sin",
    "hour_cos",

    "dow_sin",
    "dow_cos",

    "month_sin",
    "month_cos",

    "is_weekend",
    "is_peak_hour",

    "is_morning",
    "is_afternoon",
    "is_evening",
    "is_night",

    "distance_log"
]


# ==============================================================================
# 11. REMOVE INF / NAN VALUES
# ==============================================================================

df[numerical_columns] = (
    df[numerical_columns]
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
)

df = df.dropna(
    subset=numerical_columns
)

df = df.reset_index(drop=True)


# ==============================================================================
# 12. ENCODE CATEGORICAL VARIABLES
# ==============================================================================

label_encoders = {}

categorical_cardinalities = []

for column in categorical_columns:

    encoder = LabelEncoder()

    df[column] = encoder.fit_transform(
        df[column].astype(str)
    )

    label_encoders[column] = encoder

    categorical_cardinalities.append(
        len(encoder.classes_)
    )


print("\nCategorical feature sizes:")

for column, cardinality in zip(
    categorical_columns,
    categorical_cardinalities
):

    print(
        f"{column}: {cardinality}"
    )


# ==============================================================================
# 13. CREATE INPUT ARRAYS
# ==============================================================================

X_categorical = df[
    categorical_columns
].values

X_numerical = df[
    numerical_columns
].values.astype(np.float32)

y_delay = df[
    "delay_target"
].values.astype(np.int64)

y_cause = df[
    "cause_target"
].values.astype(np.int64)


# ==============================================================================
# 14. TRAIN / VALIDATION / TEST SPLIT
# ==============================================================================

(
    X_cat_temp,
    X_cat_test,

    X_num_temp,
    X_num_test,

    y_delay_temp,
    y_delay_test,

    y_cause_temp,
    y_cause_test

) = train_test_split(

    X_categorical,
    X_numerical,

    y_delay,
    y_cause,

    test_size=0.15,

    random_state=42,

    stratify=y_delay
)


(
    X_cat_train,
    X_cat_val,

    X_num_train,
    X_num_val,

    y_delay_train,
    y_delay_val,

    y_cause_train,
    y_cause_val

) = train_test_split(

    X_cat_temp,
    X_num_temp,

    y_delay_temp,
    y_cause_temp,

    test_size=0.17647,

    random_state=42,

    stratify=y_delay_temp
)


print("\nDataset split:")

print(
    "Training   :",
    len(y_delay_train)
)

print(
    "Validation :",
    len(y_delay_val)
)

print(
    "Testing    :",
    len(y_delay_test)
)


# ==============================================================================
# 15. SCALE NUMERICAL FEATURES
# ==============================================================================

# IMPORTANT:
# Fit scaler ONLY on training data.

scaler = StandardScaler()

X_num_train = scaler.fit_transform(
    X_num_train
)

X_num_val = scaler.transform(
    X_num_val
)

X_num_test = scaler.transform(
    X_num_test
)


# ==============================================================================
# 16. PYTORCH DATASET
# ==============================================================================

class BTSFlightDataset(Dataset):

    def __init__(
        self,
        categorical_data,
        numerical_data,
        delay_labels,
        cause_labels
    ):

        self.x_cat = torch.tensor(
            categorical_data,
            dtype=torch.long
        )

        self.x_num = torch.tensor(
            numerical_data,
            dtype=torch.float32
        )

        self.y_delay = torch.tensor(
            delay_labels,
            dtype=torch.long
        )

        self.y_cause = torch.tensor(
            cause_labels,
            dtype=torch.long
        )


    def __len__(self):

        return len(self.y_delay)


    def __getitem__(
        self,
        index
    ):

        return (
            self.x_cat[index],
            self.x_num[index],
            self.y_delay[index],
            self.y_cause[index]
        )


# ==============================================================================
# 17. CREATE DATASETS
# ==============================================================================

train_dataset = BTSFlightDataset(
    X_cat_train,
    X_num_train,
    y_delay_train,
    y_cause_train
)

val_dataset = BTSFlightDataset(
    X_cat_val,
    X_num_val,
    y_delay_val,
    y_cause_val
)

test_dataset = BTSFlightDataset(
    X_cat_test,
    X_num_test,
    y_delay_test,
    y_cause_test
)


# ==============================================================================
# 18. DATA LOADERS
# ==============================================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=256,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=256,
    shuffle=False,
    num_workers=0
)


# ==============================================================================
# 19. DEEP LEARNING MODEL
# ==============================================================================

class BTSDeepNeuralNetwork(nn.Module):

    def __init__(
        self,
        categorical_cardinalities,
        num_numerical_features
    ):

        super().__init__()


        # ----------------------------------------------------------------------
        # Categorical embeddings
        # ----------------------------------------------------------------------

        self.embeddings = nn.ModuleList()


        embedding_size = 0


        for cardinality in categorical_cardinalities:

            # Adaptive embedding size
            embedding_dim = min(
                32,
                max(
                    8,
                    (cardinality + 1) // 2
                )
            )

            self.embeddings.append(
                nn.Embedding(
                    num_embeddings=cardinality,
                    embedding_dim=embedding_dim
                )
            )

            embedding_size += embedding_dim


        # ----------------------------------------------------------------------
        # Total input
        # ----------------------------------------------------------------------

        total_input_size = (
            embedding_size
            +
            num_numerical_features
        )


        # ----------------------------------------------------------------------
        # Deep neural network
        # ----------------------------------------------------------------------

        self.network = nn.Sequential(

            nn.Linear(
                total_input_size,
                256
            ),

            nn.BatchNorm1d(256),

            nn.ReLU(),

            nn.Dropout(0.20),


            nn.Linear(
                256,
                128
            ),

            nn.BatchNorm1d(128),

            nn.ReLU(),

            nn.Dropout(0.20),


            nn.Linear(
                128,
                64
            ),

            nn.BatchNorm1d(64),

            nn.ReLU(),

            nn.Dropout(0.15),


            nn.Linear(
                64,
                32
            ),

            nn.BatchNorm1d(32),

            nn.ReLU()
        )


        # ----------------------------------------------------------------------
        # Delay prediction
        # ----------------------------------------------------------------------

        self.delay_output = nn.Linear(
            32,
            2
        )


        # ----------------------------------------------------------------------
        # Cause prediction
        # ----------------------------------------------------------------------

        self.cause_output = nn.Linear(
            32,
            5
        )


    def forward(
        self,
        x_categorical,
        x_numerical
    ):

        embedded_features = []


        for i in range(
            x_categorical.shape[1]
        ):

            embedding = self.embeddings[i](
                x_categorical[:, i]
            )

            embedded_features.append(
                embedding
            )


        categorical_vector = torch.cat(
            embedded_features,
            dim=1
        )


        combined_input = torch.cat(
            [
                categorical_vector,
                x_numerical
            ],
            dim=1
        )


        features = self.network(
            combined_input
        )


        delay_logits = self.delay_output(
            features
        )

        cause_logits = self.cause_output(
            features
        )


        return (
            delay_logits,
            cause_logits
        )


# ==============================================================================
# 20. INITIALIZE MODEL
# ==============================================================================

model = BTSDeepNeuralNetwork(

    categorical_cardinalities=
        categorical_cardinalities,

    num_numerical_features=
        len(numerical_columns)

).to(device)


print("\n================================================")
print("IMPROVED DEEP LEARNING MODEL")
print("================================================")

print(model)


# ==============================================================================
# 21. LOSS FUNCTIONS
# ==============================================================================

# We intentionally use normal CrossEntropyLoss for the
# main delay target because the requested metric is accuracy.
#
# Class-weighting can improve minority recall but can sometimes
# reduce overall accuracy on an imbalanced dataset.

delay_loss_function = nn.CrossEntropyLoss()

cause_loss_function = nn.CrossEntropyLoss()


# Cause prediction is auxiliary.
# A smaller weight prevents cause prediction from hurting
# the main delay-prediction task.

CAUSE_LOSS_WEIGHT = 0.25


# ==============================================================================
# 22. OPTIMIZER
# ==============================================================================

optimizer = optim.AdamW(

    model.parameters(),

    lr=0.0005,

    weight_decay=1e-5
)


# ==============================================================================
# 23. LEARNING RATE SCHEDULER
# ==============================================================================

scheduler = optim.lr_scheduler.ReduceLROnPlateau(

    optimizer,

    mode="max",

    factor=0.5,

    patience=4
)


# ==============================================================================
# 24. VALIDATION FUNCTION
# ==============================================================================

def evaluate_accuracy(
    model,
    data_loader
):

    model.eval()

    predictions = []

    actuals = []

    probabilities = []


    with torch.no_grad():

        for (
            batch_cat,
            batch_num,
            batch_delay,
            batch_cause
        ) in data_loader:

            batch_cat = batch_cat.to(device)

            batch_num = batch_num.to(device)


            delay_logits, _ = model(
                batch_cat,
                batch_num
            )


            probability = torch.softmax(
                delay_logits,
                dim=1
            )[:, 1]


            prediction = torch.argmax(
                delay_logits,
                dim=1
            )


            predictions.extend(
                prediction.cpu().numpy()
            )

            actuals.extend(
                batch_delay.numpy()
            )

            probabilities.extend(
                probability.cpu().numpy()
            )


    accuracy = accuracy_score(
        actuals,
        predictions
    )


    return (
        accuracy,
        np.array(actuals),
        np.array(predictions),
        np.array(probabilities)
    )


# ==============================================================================
# 25. TRAINING
# ==============================================================================

epochs = 75

best_validation_accuracy = 0.0

best_model_state = None

patience = 12

epochs_without_improvement = 0


print("\n================================================")
print("TRAINING DEEP NEURAL NETWORK")
print("================================================")


for epoch in range(
    1,
    epochs + 1
):

    model.train()

    total_loss = 0.0


    for (
        batch_cat,
        batch_num,
        batch_delay,
        batch_cause
    ) in train_loader:


        batch_cat = batch_cat.to(device)

        batch_num = batch_num.to(device)

        batch_delay = batch_delay.to(device)

        batch_cause = batch_cause.to(device)


        optimizer.zero_grad()


        # ----------------------------------------------------------------------
        # Forward propagation
        # ----------------------------------------------------------------------

        delay_logits, cause_logits = model(
            batch_cat,
            batch_num
        )


        # ----------------------------------------------------------------------
        # Delay loss
        # ----------------------------------------------------------------------

        delay_loss = delay_loss_function(
            delay_logits,
            batch_delay
        )


        # ----------------------------------------------------------------------
        # Cause loss
        # ----------------------------------------------------------------------

        cause_loss = cause_loss_function(
            cause_logits,
            batch_cause
        )


        # ----------------------------------------------------------------------
        # Multi-task loss
        # ----------------------------------------------------------------------

        total_batch_loss = (

            delay_loss

            +

            CAUSE_LOSS_WEIGHT
            *
            cause_loss
        )


        # ----------------------------------------------------------------------
        # Backpropagation
        # ----------------------------------------------------------------------

        total_batch_loss.backward()


        # Prevent exploding gradients

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )


        optimizer.step()


        total_loss += (
            total_batch_loss.item()
        )


    average_loss = (
        total_loss /
        len(train_loader)
    )


    # --------------------------------------------------------------------------
    # Validation
    # --------------------------------------------------------------------------

    validation_accuracy, _, _, _ = evaluate_accuracy(
        model,
        val_loader
    )


    scheduler.step(
        validation_accuracy
    )


    current_lr = optimizer.param_groups[0]["lr"]


    print(
        f"Epoch [{epoch:02d}/{epochs}] "
        f"Loss: {average_loss:.4f} "
        f"Validation Accuracy: "
        f"{validation_accuracy:.4f} "
        f"LR: {current_lr:.7f}"
    )


    # --------------------------------------------------------------------------
    # Save best model
    # --------------------------------------------------------------------------

    if validation_accuracy > best_validation_accuracy:

        best_validation_accuracy = (
            validation_accuracy
        )

        best_model_state = {
            key: value.detach().cpu().clone()
            for key, value in model.state_dict().items()
        }

        epochs_without_improvement = 0

    else:

        epochs_without_improvement += 1


    # --------------------------------------------------------------------------
    # Early stopping
    # --------------------------------------------------------------------------

    if epochs_without_improvement >= patience:

        print(
            "\nEarly stopping activated."
        )

        break


# ==============================================================================
# 26. LOAD BEST MODEL
# ==============================================================================

if best_model_state is not None:

    model.load_state_dict(
        best_model_state
    )

    model = model.to(device)


print("\nBest validation accuracy:")

print(
    f"{best_validation_accuracy:.4f}"
)


# ==============================================================================
# 27. FIND BEST CLASSIFICATION THRESHOLD
# ==============================================================================

print("\n================================================")
print("OPTIMIZING DECISION THRESHOLD")
print("================================================")


_, val_actual, _, val_probability = (
    evaluate_accuracy(
        model,
        val_loader
    )
)


best_threshold = 0.50

best_threshold_accuracy = 0.0


# Search thresholds

for threshold in np.arange(
    0.20,
    0.81,
    0.01
):

    threshold_prediction = (
        val_probability >= threshold
    ).astype(int)


    threshold_accuracy = accuracy_score(
        val_actual,
        threshold_prediction
    )


    if threshold_accuracy > best_threshold_accuracy:

        best_threshold_accuracy = (
            threshold_accuracy
        )

        best_threshold = threshold


print(
    f"Best Threshold       : "
    f"{best_threshold:.2f}"
)

print(
    f"Validation Accuracy  : "
    f"{best_threshold_accuracy:.4f}"
)


# ==============================================================================
# 28. FINAL TEST EVALUATION
# ==============================================================================

print("\n================================================")
print("FINAL DEEP LEARNING TEST")
print("================================================")


_, actual_delay, _, predicted_probability = (
    evaluate_accuracy(
        model,
        test_loader
    )
)


# Use optimized threshold

predicted_delay = (
    predicted_probability >= best_threshold
).astype(int)


# ==============================================================================
# 29. CALCULATE METRICS
# ==============================================================================

accuracy = accuracy_score(
    actual_delay,
    predicted_delay
)

precision = precision_score(
    actual_delay,
    predicted_delay,
    zero_division=0
)

recall = recall_score(
    actual_delay,
    predicted_delay,
    zero_division=0
)

f1 = f1_score(
    actual_delay,
    predicted_delay,
    zero_division=0
)

auc = roc_auc_score(
    actual_delay,
    predicted_probability
)


# ==============================================================================
# 30. RESULTS
# ==============================================================================

print("\n================================================")
print("IMPROVED DEEP LEARNING MODEL RESULTS")
print("================================================")

print(
    f"Accuracy  : {accuracy:.4f}"
)

print(
    f"Precision : {precision:.4f}"
)

print(
    f"Recall    : {recall:.4f}"
)

print(
    f"F1 Score  : {f1:.4f}"
)

print(
    f"ROC-AUC   : {auc:.4f}"
)

print(
    f"Threshold : {best_threshold:.2f}"
)

print("================================================")


# ==============================================================================
# 31. CLASSIFICATION REPORT
# ==============================================================================

print("\nDelay Classification Report:")

print(
    classification_report(

        actual_delay,

        predicted_delay,

        target_names=[
            "On Time",
            "Delayed"
        ],

        zero_division=0
    )
)


# ==============================================================================
# 32. CONFUSION MATRIX
# ==============================================================================

print("\n================================================")
print("CONFUSION MATRIX")
print("================================================")

cm = confusion_matrix(
    actual_delay,
    predicted_delay
)

print(cm)

print("\nInterpretation:")

print(
    "True On Time  :",
    cm[0][0]
)

print(
    "False Delayed :",
    cm[0][1]
)

print(
    "False On Time :",
    cm[1][0]
)

print(
    "True Delayed  :",
    cm[1][1]
)


# ==============================================================================
# 33. DELAY CAUSE PREDICTION
# ==============================================================================

model.eval()

predicted_cause = []

actual_cause = []


with torch.no_grad():

    for (
        batch_cat,
        batch_num,
        batch_delay,
        batch_cause
    ) in test_loader:


        batch_cat = batch_cat.to(device)

        batch_num = batch_num.to(device)


        _, cause_logits = model(
            batch_cat,
            batch_num
        )


        cause_prediction = torch.argmax(
            cause_logits,
            dim=1
        )


        predicted_cause.extend(
            cause_prediction.cpu().numpy()
        )

        actual_cause.extend(
            batch_cause.numpy()
        )


# ==============================================================================
# 34. CAUSE CLASSIFICATION
# ==============================================================================

cause_names = [

    "Carrier",
    "Weather",
    "NAS",
    "Security",
    "Late Aircraft"
]


print("\n================================================")
print("DELAY CAUSE CLASSIFICATION")
print("================================================")


print(
    classification_report(

        actual_cause,

        predicted_cause,

        labels=[
            0,
            1,
            2,
            3,
            4
        ],

        target_names=cause_names,

        zero_division=0
    )
)


# ==============================================================================
# 35. SAMPLE PREDICTION
# ==============================================================================

sample_cat = torch.tensor(
    X_cat_test[:1],
    dtype=torch.long
).to(device)


sample_num = torch.tensor(
    X_num_test[:1],
    dtype=torch.float32
).to(device)


model.eval()


with torch.no_grad():

    delay_logits, cause_logits = model(
        sample_cat,
        sample_num
    )


delay_probability = torch.softmax(
    delay_logits,
    dim=1
)[0, 1].item()


predicted_delay_class = (

    1
    if delay_probability >= best_threshold
    else 0
)


predicted_cause_class = torch.argmax(
    cause_logits,
    dim=1
).item()


print("\n================================================")
print("SAMPLE FLIGHT PREDICTION")
print("================================================")

print(
    "Delay Probability:",
    f"{delay_probability * 100:.2f}%"
)

print(
    "Decision Threshold:",
    f"{best_threshold * 100:.2f}%"
)


if predicted_delay_class == 1:

    print(
        "Predicted Status: DELAYED"
    )

    print(
        "Predicted Cause:",
        cause_names[
            predicted_cause_class
        ]
    )

else:

    print(
        "Predicted Status: ON TIME"
    )

    print(
        "Predicted Cause: None"
    )


print("================================================")


# ==============================================================================
# 36. SAVE MODEL
# ==============================================================================

torch.save(

    {

        "model_state_dict":
            model.state_dict(),

        "categorical_cardinalities":
            categorical_cardinalities,

        "numerical_columns":
            numerical_columns,

        "categorical_columns":
            categorical_columns,

        "best_threshold":
            best_threshold,

        "scaler":
            scaler,

        "label_encoders":
            label_encoders

    },

    "bts_improved_deep_learning_model.pth"
)


print(
    "\nModel saved as:"
)

print(
    "bts_improved_deep_learning_model.pth"
)


# ==============================================================================
# 37. FINAL SUMMARY
# ==============================================================================

print("\n================================================")
print("FINAL SUMMARY")
print("================================================")

print(
    f"Test Accuracy  : {accuracy * 100:.2f}%"
)

print(
    f"Test Precision : {precision * 100:.2f}%"
)

print(
    f"Test Recall    : {recall * 100:.2f}%"
)

print(
    f"Test F1 Score  : {f1 * 100:.2f}%"
)

print(
    f"Test ROC-AUC   : {auc:.4f}"
)

print(
    f"Best Threshold : {best_threshold:.2f}"
)

print("================================================")

Using device: cpu

Loading dataset:
final_dataset_12_columns.csv
Original dataset shape: (357840, 12)

Cleaning dataset...
Dataset shape after cleaning: (357840, 12)

Delay distribution:
delay_target
1    280785
0     77055
Name: count, dtype: int64

Delay percentages:
delay_target
1    78.466633
0    21.533367
Name: proportion, dtype: float64

Creating features...

Categorical feature sizes:
AIRLINE: 18
ORIGIN: 379
DEST: 377
ROUTE: 7212

Dataset split:
Training   : 250488
Validation : 53676
Testing    : 53676

IMPROVED DEEP LEARNING MODEL
BTSDeepNeuralNetwork(
  (embeddings): ModuleList(
    (0): Embedding(18, 9)
    (1): Embedding(379, 32)
    (2): Embedding(377, 32)
    (3): Embedding(7212, 32)
  )
  (network): Sequential(
    (0): Linear(in_features=128, out_features=256, bias=True)
    (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=256, out_features=128, bias=T